# Building conversational applications with CrewAI

In this notebook, you learn how to use CrewAI to create collaborative AI agents that can work together to solve complex tasks.

This demonstrates the same functionality as the Strands Agents example, but using CrewAI's multi-agent approach.

## Environment setup

In this task, you set up your environment and install the required packages for CrewAI.

In [1]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

model_id = "amazon.nova-lite-v1:0"

## Define Tools for CrewAI

In this task, we define the tools that our CrewAI agents will use.

In [2]:
@tool
def calculator_tool(oper1: int, oper2: int, operator: str) -> str:
    """
    Perform basic arithmetic operations on two numbers.
    
    Args:
        oper1: First operand
        oper2: Second operand  
        operator: Operation to perform (+, -, *, / or add, subtract, multiply, divide)
    
    Returns:
        Result of the calculation as a string
    """
    def add(a, b):
        return a + b

    def subtract(a, b):
        return a - b

    def multiply(a, b):
        return a * b

    def divide(a, b):
        if b == 0:
            return "Cannot divide by zero"
        else:
            return float(a / b)
    
    if operator == '+' or operator == 'add':
        return str(add(oper1, oper2))
    elif operator == '-' or operator == 'subtract':
        return str(subtract(oper1, oper2))
    elif operator == '*' or operator == 'multiply':
        return str(multiply(float(oper1), float(oper2)))
    elif operator == '/' or operator == 'divide':
        return str(divide(oper1, oper2))
    else:
        return "Invalid operator"

In [3]:
@tool
def search_tool(query: str) -> str:
    """
    Search with DuckDuckGo for information about a topic.
    
    Args:
        query: The search query
    
    Returns:
        DuckDuckGo search results as a string
    """
    tool = DuckDuckGoSearchRun()
    result = tool.invoke(query)
    return result

## Create CrewAI Agents and Tasks

Now we create specialized agents for different tasks and define how they work together.

In [4]:
# Create the Bedrock LLM
llm = LLM(
    model="bedrock/us.amazon.nova-lite-v1:0"
)

# Create agent
agent = Agent(
    role="Expert in searching and calculations",
    goal="Use available tools to find Find accurate information on any topic using tools",
    backstory="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo. Use the available tools when needed to provide accurate and helpful responses.",
    tools=[search_tool, calculator_tool],
    llm=llm,
    verbose=True
)

## Test the CrewAI System

Now let's test our CrewAI system with the same queries used in the Strands example.

In [5]:
def run_crew_task(user_query: str):
    """Run a CrewAI task based on user query"""
    
    # Create task
    task = Task(
        description=f"Answer this query: {user_query}",
        expected_output="A comprehensive answer to the user's query",
        agent=agent
    )
    
    # Create crew
    crew = Crew(
        agents=[agent],
        tasks=[task],
        verbose=True
    )
    
    # Execute
    result = crew.kickoff()
    return result

In [6]:
# Test with the same complex query from Strands example
query = "What is Amazon SageMaker? What is launch year multiplied by 2"

print(f"Query: {query}\n")
print("CrewAI Response:")
print("=" * 50)

response = run_crew_task(query)
print(response)

Query: What is Amazon SageMaker? What is launch year multiplied by 2

CrewAI Response:


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.28                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cfdfc2e2-ae1b-461a-8397-6249c8980c58                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│  ID: 819e1c43-15ee-4308-83fd-d342b2749e6c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Task: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Amazon SageMaker is a fully managed service designed to simplify the process of building, training, and        │
│  deploying machine learning (ML) models. It is created to support enterprises on their machine learning         │
│  journey, making it efficient and easy. The launch year of Amazon SageMaker was 2017, and when multiplied by    │
│  2, the result is 4034.                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this query: What is Amazon SageMaker? What is launch year multiplied by 2                         │
│  Agent: Expert in searching and calculations                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cfdfc2e2-ae1b-461a-8397-6249c8980c58                                                                       │
│  Final Output: Amazon SageMaker is a fully managed service designed to simplify the process of building,        │
│  training, and deploying machine learning (ML) models. It is created to support enterprises on their machine    │
│  learning journey, making it efficient and easy. The launch year of Amazon SageMaker was 2017, and when         │
│  multiplied by 2, the result is 4034.                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Amazon SageMaker is a fully managed service designed to simplify the process of building, training, and deploying machine learning (ML) models. It is created to support enterprises on their machine learning journey, making it efficient and easy. The launch year of Amazon SageMaker was 2017, and when multiplied by 2, the result is 4034.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Test with calculation-only query
calc_query = "What is 156 multiplied by 23?"
print(f"Query: {calc_query}")
print(f"Response: {run_crew_task(calc_query)}")
print("\n" + "="*50 + "\n")

In [ ]:
# Test with search-only query
search_query = "Tell me about artificial intelligence"
print(f"Query: {search_query}")
print(f"Response: {run_crew_task(search_query)}")
print("\n" + "="*50 + "\n")